In [1]:
import xarray as xr
import numpy as np

In [2]:
f = xr.open_dataset("/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/prra_SPEEDY_1990.nc")
f

<xarray.Dataset> Size: 54MB
Dimensions:    (time: 2920, lat: 48, lon: 96, bnds: 2)
Coordinates:
  * time       (time) datetime64[ns] 23kB 1990-01-01T01:30:00 ... 1990-12-31T...
  * lat        (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon        (lon) float64 768B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
  * bnds       (bnds) int64 16B 0 1
Data variables:
    prra       (time, lat, lon) float32 54MB ...
    time_bnds  (time, bnds) datetime64[ns] 47kB ...
Attributes:
    Conventions:  CF-1.7
    title:        SPEEDY forcing for ACCESS-OM2
    source:       SPEEDY model output
    frequency:    3hr
    history:      Created from SPEEDY PRECLS, PRECNV and SNOW and exported as...
    comment:      3-hourly mean liquid precipitation forcing following JRA55-...

In [3]:
lat = f.lat.values
lon = f.lon.values

In [4]:
nodes, w = np.polynomial.legendre.leggauss(len(lat))
glat = np.degrees(np.arcsin(nodes))

print("shape:", len(lat), len(lon))
print("lat first:", lat[:6])
print("lat last :", lat[-6:])
print("lon first:", lon[:6])
print("lon last :", lon[-6:])
print("max Gaussian latitude difference:",
      np.max(np.abs(np.sort(lat) - glat)))

shape: 48 96
lat first: [-87.159 -83.479 -79.777 -76.07  -72.362 -68.652]
lat last : [68.652 72.362 76.07  79.777 83.479 87.159]
lon first: [ 0.    3.75  7.5  11.25 15.   18.75]
lon last : [337.5  341.25 345.   348.75 352.5  356.25]
max Gaussian latitude difference: 0.0004767159783369834


In [5]:
import sys

ESMGRIDS = "/leonardo/home/userexternal/ntilinin/COSIMA/access-om2/tools/esmgrids"
sys.path.insert(0, ESMGRIDS)

from esmgrids.speedy_grid import SpeedyGrid

print("SpeedyGrid imported")

SpeedyGrid imported


In [6]:
SPEEDY_FILE = ("/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/prra_SPEEDY_1990.nc")

ds = xr.open_dataset(SPEEDY_FILE)
ds

<xarray.Dataset> Size: 54MB
Dimensions:    (time: 2920, lat: 48, lon: 96, bnds: 2)
Coordinates:
  * time       (time) datetime64[ns] 23kB 1990-01-01T01:30:00 ... 1990-12-31T...
  * lat        (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon        (lon) float64 768B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
  * bnds       (bnds) int64 16B 0 1
Data variables:
    prra       (time, lat, lon) float32 54MB ...
    time_bnds  (time, bnds) datetime64[ns] 47kB ...
Attributes:
    Conventions:  CF-1.7
    title:        SPEEDY forcing for ACCESS-OM2
    source:       SPEEDY model output
    frequency:    3hr
    history:      Created from SPEEDY PRECLS, PRECNV and SNOW and exported as...
    comment:      3-hourly mean liquid precipitation forcing following JRA55-...

In [7]:
print("dims:", ds.sizes)
print("lat:", ds.lat.values[:5], "...", ds.lat.values[-5:])
print("lon:", ds.lon.values[:5], "...", ds.lon.values[-5:])

dims: Frozen({'time': 2920, 'lat': 48, 'lon': 96, 'bnds': 2})
lat: [-87.159 -83.479 -79.777 -76.07  -72.362] ... [72.362 76.07  79.777 83.479 87.159]
lon: [ 0.    3.75  7.5  11.25 15.  ] ... [341.25 345.   348.75 352.5  356.25]


In [8]:
from pathlib import Path

ESMGRIDS_SRC = Path(
    "/leonardo/home/userexternal/ntilinin/COSIMA/access-om2/tools/esmgrids/esmgrids"
)

for f in ESMGRIDS_SRC.glob("*.py"):
    text = f.read_text()
    if "np.NAN" in text:
        print(f.name, text.count("np.NAN"))

In [9]:
for f in ESMGRIDS_SRC.glob("*.py"):
    text = f.read_text()
    if "np.NAN" in text:
        f.write_text(text.replace("np.NAN", "np.nan"))
        print("fixed:", f.name)

In [10]:
g = SpeedyGrid(str(SPEEDY_FILE))

print("shape:", g.x_t.shape)
print("nlat:", g.num_lat_points)
print("nlon:", g.num_lon_points)

print("lat first:", g.y_t[:6, 0])
print("lat last :", g.y_t[-6:, 0])

print("lon first:", g.x_t[0, :6])
print("lon last :", g.x_t[0, -6:])

shape: (48, 96)
nlat: 48
nlon: 96
lat first: [-87.159 -83.479 -79.777 -76.07  -72.362 -68.652]
lat last : [68.652 72.362 76.07  79.777 83.479 87.159]
lon first: [ 0.    3.75  7.5  11.25 15.   18.75]
lon last : [337.5  341.25 345.   348.75 352.5  356.25]


In [11]:
assert np.allclose(g.y_t[:, 0], ds.lat.values)
assert np.allclose(g.x_t[0, :], ds.lon.values)

print("Forcing coordinates and SpeedyGrid coordinates match.")

Forcing coordinates and SpeedyGrid coordinates match.


In [12]:
# North boundary of row j must equal South boundary of row j+1
print("lat connectivity:",
      np.max(np.abs(g.clat_t[2, :-1, :] - g.clat_t[0, 1:, :])))

# East boundary of column i must equal West boundary of column i+1
print("lon connectivity:",
      np.max(np.abs(g.clon_t[1, :, :-1] - g.clon_t[0, :, 1:])))

print("first cell lat corners:", g.clat_t[:, 0, 0])
print("first cell lon corners:", g.clon_t[:, 0, 0])

print("last lon cell corners:", g.clon_t[:, 0, -1])

lat connectivity: 0.0
lon connectivity: 0.0
first cell lat corners: [-90.         -90.         -85.44867722 -85.44867722]
first cell lon corners: [-1.875  1.875  1.875 -1.875]
last lon cell corners: [354.375 358.125 358.125 354.375]


In [13]:
west_first = g.clon_t[0, :, 0]
east_last = g.clon_t[1, :, -1]

print("raw seam difference:",
      np.unique(west_first - east_last))

print("periodic seam difference:",
      np.unique(((west_first - east_last + 180) % 360) - 180))

raw seam difference: [-360.]
periodic seam difference: [0.]


In [14]:
from esmgrids.jra55_grid import Jra55Grid

JRA_FILE = "/leonardo_scratch/fast/ICT26_ESP/ntilinin/make_ryf/RYF.tas.1990_1991.nc"

jg = Jra55Grid(JRA_FILE)

print("JRA first lon center:", jg.x_t[0, 0])
print("JRA last lon center :", jg.x_t[0, -1])

print("JRA first cell corners:",
      jg.clon_t[:, 0, 0])

print("JRA last cell corners:",
      jg.clon_t[:, 0, -1])

JRA first lon center: 0.0
JRA last lon center : 359.4375
JRA first cell corners: [-0.28125  0.28125  0.28125 -0.28125]
JRA last cell corners: [359.15625 359.71875 359.71875 359.15625]


In [12]:
nodes, weights = np.polynomial.legendre.leggauss(48)
gaussian_lat = np.degrees(np.arcsin(nodes))

print("max Gaussian latitude difference:",
      np.max(np.abs(ds.lat.values - gaussian_lat)))

max Gaussian latitude difference: 0.0004767159783369834


In [15]:
print("JRA first lat cell:", jg.clat_t[:, 0, 0])
print("JRA second lat cell:", jg.clat_t[:, 1, 0])
print("JRA last lat cell :", jg.clat_t[:, -1, 0])

JRA first lat cell: [-90.         -90.         -89.29163284 -89.29163284]
JRA second lat cell: [-89.29163284 -89.29163284 -88.73307498 -88.73307498]
JRA last lat cell : [89.29163284 89.29163284 90.         90.        ]


In [16]:
from pathlib import Path
import sys
import xarray as xr
import numpy as np

sys.path.insert(
    0,
    "/leonardo/home/userexternal/ntilinin/COSIMA/access-om2/tools/esmgrids"
)

from esmgrids.mom_grid import MomGrid

HGRID = "/leonardo_scratch/fast/ICT26_ESP/ntilinin/INPUT/access-om2/ocean/grids/mosaic/global.1deg/2020.05.30/ocean_hgrid.nc"
MASK = "/leonardo_scratch/fast/ICT26_ESP/ntilinin/INPUT/access-om2/ocean/grids/bathymetry/global.1deg/2020.10.22/ocean_mask.nc"

OLD_W = "/leonardo_scratch/fast/ICT26_ESP/ntilinin/INPUT/access-om2/remapping_weights/JRA55/global.1deg/2020.05.30/rmp_jra55_cice_1st_conserve.nc"

mom = MomGrid.fromfile(HGRID, mask_file=MASK)

TEST = "/tmp/mom1_scrip_test.nc"
mom.write_scrip(TEST, write_test_scrip=False)

new = xr.open_dataset(TEST)
old = xr.open_dataset(OLD_W)

print("new dims:", new.grid_dims.values)
print("old dims:", old.dst_grid_dims.values)

print("center lat max diff:",
      np.max(np.abs(new.grid_center_lat.values -
                    old.dst_grid_center_lat.values)))

dlon = ((new.grid_center_lon.values -
         old.dst_grid_center_lon.values + 180) % 360) - 180

print("center lon max diff:", np.max(np.abs(dlon)))

print("corner lat max diff:",
      np.max(np.abs(new.grid_corner_lat.values -
                    old.dst_grid_corner_lat.values)))

dlon = ((new.grid_corner_lon.values -
         old.dst_grid_corner_lon.values + 180) % 360) - 180

print("corner lon max diff:", np.max(np.abs(dlon)))

print("mask equal:",
      np.array_equal(new.grid_imask.values,
                     old.dst_grid_imask.values))

new dims: [360 300]
old dims: [360 300]
center lat max diff: 0.0
center lon max diff: 0.0
corner lat max diff: 0.0
corner lon max diff: 0.0
mask equal: True


In [ ]:
from esmgrids.jra55_grid import Jra55Grid
import xarray as xr
import numpy as np

JRA_FILE = "/leonardo_scratch/fast/ICT26_ESP/ntilinin/make_ryf/RYF.tas.1990_1991.nc"

jg = Jra55Grid(JRA_FILE)

JRA_SCRIP = "/tmp/jra55_scrip_test.nc"
jg.write_scrip(JRA_SCRIP, mask=np.zeros_like(jg.mask_t), write_test_scrip=False)

new = xr.open_dataset(JRA_SCRIP)
old = xr.open_dataset(OLD_W)

print("new dims:", new.grid_dims.values)
print("old dims:", old.src_grid_dims.values)

print("center lat max diff:",
      np.max(np.abs(new.grid_center_lat.values -
                    old.src_grid_center_lat.values)))

dlon = ((new.grid_center_lon.values -
         old.src_grid_center_lon.values + 180) % 360) - 180
print("center lon max diff:", np.max(np.abs(dlon)))

print("corner lat max diff:",
      np.max(np.abs(new.grid_corner_lat.values -
                    old.src_grid_corner_lat.values)))

dlon = ((new.grid_corner_lon.values -
         old.src_grid_corner_lon.values + 180) % 360) - 180
print("corner lon max diff:", np.max(np.abs(dlon)))

print("mask equal:",
      np.array_equal(new.grid_imask.values,
                     old.src_grid_imask.values))

In [13]:
print("corner array shape:", g.clat_t.shape)

print("southern boundary:",
      g.clat_t[0, 0, 0],
      g.clat_t[1, 0, 0])

print("northern boundary:",
      g.clat_t[2, -1, 0],
      g.clat_t[3, -1, 0])

print("first cell corners lat:",
      g.clat_t[:, 0, 0])

print("first cell corners lon:",
      g.clon_t[:, 0, 0])

corner array shape: (4, 48, 96)
southern boundary: -90.0 -90.0
northern boundary: 90.0 90.0
first cell corners lat: [-90.         -90.         -85.44867722 -85.44867722]
first cell corners lon: [-1.875  1.875  1.875 -1.875]


In [14]:
assert g.x_t.shape == (48, 96)
assert g.y_t.shape == (48, 96)
assert g.clat_t.shape == (4, 48, 96)
assert g.clon_t.shape == (4, 48, 96)

assert np.isfinite(g.x_t).all()
assert np.isfinite(g.y_t).all()
assert np.isfinite(g.clat_t).all()
assert np.isfinite(g.clon_t).all()
assert np.isfinite(g.area_t).all()

assert np.all(np.diff(g.y_t[:, 0]) > 0)
assert np.all(np.diff(g.x_t[0, :]) > 0)

assert np.isclose(g.clat_t[0, 0, 0], -90.0)
assert np.isclose(g.clat_t[2, -1, 0], 90.0)

print("All SPEEDY grid checks passed.")

All SPEEDY grid checks passed.


In [15]:
SCRIP_FILE = Path("speedy_t30_scrip.nc")

g.write_scrip(
    str(SCRIP_FILE),
    mask=np.zeros_like(g.mask_t, dtype=int),
    write_test_scrip=False
)

print(SCRIP_FILE)

speedy_t30_scrip.nc


In [16]:
scrip = xr.open_dataset(SCRIP_FILE)

print(scrip.grid_dims.values)
print(scrip.grid_center_lat.shape)
print(scrip.grid_corner_lat.shape)

scrip

[96 48]
(4608,)
(4608, 4)


<xarray.Dataset> Size: 387kB
Dimensions:          (grid_rank: 2, grid_size: 4608, grid_corners: 4)
Dimensions without coordinates: grid_rank, grid_size, grid_corners
Data variables:
    grid_dims        (grid_rank) int32 8B 96 48
    grid_center_lat  (grid_size) float64 37kB ...
    grid_center_lon  (grid_size) float64 37kB ...
    grid_imask       (grid_size) int32 18kB ...
    grid_corner_lat  (grid_size, grid_corners) float64 147kB ...
    grid_corner_lon  (grid_size, grid_corners) float64 147kB ...
Attributes:
    title:    SPEEDY T30 Gaussian grid
    history:

In [17]:
import shutil

for exe in ["ESMF_RegridWeightGen", "mpirun", "ncrename"]:
    print(exe, "->", shutil.which(exe))

ESMF_RegridWeightGen -> None
mpirun -> None
ncrename -> None


In [18]:
from pathlib import Path
from esmgrids.mom_grid import MomGrid

ACCESS_INPUT = Path(
    "/leonardo_scratch/fast/ICT26_ESP/ntilinin/INPUT/access-om2"
)

OCEAN_HGRID = ACCESS_INPUT / "ocean/grids/mosaic/global.1deg/2020.05.30/ocean_hgrid.nc"
OCEAN_MASK  = ACCESS_INPUT / "ocean/grids/bathymetry/global.1deg/2020.10.22/ocean_mask.nc"

dest_grid = MomGrid.fromfile(
    str(OCEAN_HGRID),
    mask_file=str(OCEAN_MASK)
)

print("SPEEDY:", g.num_lon_points, g.num_lat_points)
print("MOM/CICE:", dest_grid.num_lon_points, dest_grid.num_lat_points)

SPEEDY: 96 48
MOM/CICE: 360 300


In [19]:
import sys

TOOLS = "/leonardo/home/userexternal/ntilinin/COSIMA/access-om2/tools"
if TOOLS not in sys.path:
    sys.path.append(TOOLS)

from make_remap_weights import create_weights, convert_to_scrip_output

In [21]:
import shutil

for exe in ["mpirun", "mpiexec", "srun", "ESMF_RegridWeightGen", "ncrename"]:
    print(exe, "->", shutil.which(exe))

mpirun -> None
mpiexec -> None
srun -> /usr/bin/srun
ESMF_RegridWeightGen -> None
ncrename -> None


In [20]:
weights = create_weights(
    g,
    dest_grid,
    npes=4,
    method="conserve"
)

print(weights)

['mpirun', '-np', '4', 'ESMF_RegridWeightGen', '--netcdf4', '-s', '/leonardo/home/userexternal/ntilinin/COSIMA/access-om2/tools/tmpg9h2qvx6.nc', '-d', '/leonardo/home/userexternal/ntilinin/COSIMA/access-om2/tools/tmpyk0waw6o.nc', '-m', 'conserve', '-w', '/leonardo/home/userexternal/ntilinin/COSIMA/access-om2/tools/tmpwygdxhm9.nc']


FileNotFoundError: [Errno 2] No such file or directory: 'mpirun'